# Final Control Strategy and Models (Person 2 — Ruben)

Updated to align with the locked-in variable conventions from notebook 02:
the financial literacy measures are now `finlit_z_index` (primary) and
`finlit_pca1` (alternative). This notebook finalizes the control strategy,
runs diagnostics (VIF and a missing-data/selection check), and re-estimates
the adjusted models on both the full (800) and deduplicated (555) samples.

In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", None)

In [13]:
# Load raw data and build the two parallel samples used throughout the project
df_full = pd.read_csv("../data/Clark_County_Senior_FIS_Data_N_800.csv")
df_unique = df_full.drop_duplicates().copy()
df_full   = df_full.copy()

print("Full sample:", df_full.shape)
print("Deduplicated sample:", df_unique.shape)

Full sample: (800, 112)
Deduplicated sample: (555, 112)


In [14]:
# --- Outcome variables (USDA six-item scoring) ---
def construct_food_security_outcomes(df):
    df = df.copy()
    df["fs_q17_affirm"] = df["q17"].isin([1, 2]).astype(int)
    df["fs_q18_affirm"] = df["q18"].isin([1, 2]).astype(int)
    df["fs_q19_affirm"] = (df["q19"] == 1).astype(int)
    df["fs_q20_affirm"] = df["q20"].isin([1, 2]).astype(int)
    df["fs_q21_affirm"] = (df["q21"] == 1).astype(int)
    df["fs_q22_affirm"] = (df["q22"] == 1).astype(int)
    items = ["fs_q17_affirm","fs_q18_affirm","fs_q19_affirm",
             "fs_q20_affirm","fs_q21_affirm","fs_q22_affirm"]
    df["food_security_raw_score"] = df[items].sum(axis=1)
    df["food_insecure"]     = (df["food_security_raw_score"] >= 2).astype(int)
    df["very_low_security"] = (df["food_security_raw_score"] >= 5).astype(int)
    return df

# --- Financial literacy measures (aligned with notebook 02) ---
q45_items   = ["q45a","q45b","q45c","q45d","q45e","q45f","q45g","q45h"]
zscore_vars = ["q44"] + q45_items

# Primary: z-score each item, then average (matches Chad's finlit_z_index)
def construct_zscore_finlit_index(df):
    df = df.copy()
    z = StandardScaler().fit_transform(df[zscore_vars])
    z_df = pd.DataFrame(z, columns=[f"{c}_z" for c in zscore_vars], index=df.index)
    df = pd.concat([df, z_df], axis=1)
    df["finlit_z_index"] = z_df.mean(axis=1)
    return df

# Alternative: first principal component of the same items
def construct_pca_finlit(df):
    df = df.copy()
    X_scaled = StandardScaler().fit_transform(df[zscore_vars])
    pca = PCA(n_components=1)
    comp = pca.fit_transform(X_scaled)[:, 0]
    if pca.components_[0].mean() < 0:   # keep orientation: higher = stronger finlit
        comp = -comp
    df["finlit_pca1"] = comp
    return df

df_full   = construct_pca_finlit(construct_zscore_finlit_index(construct_food_security_outcomes(df_full)))
df_unique = construct_pca_finlit(construct_zscore_finlit_index(construct_food_security_outcomes(df_unique)))

# Sanity check — should match notebook 02
for name, d in [("Full (800)", df_full), ("Dedup (555)", df_unique)]:
    print(f"{name}: food_insecure={d['food_insecure'].mean():.1%}, "
          f"very_low_security={d['very_low_security'].mean():.1%}, "
          f"z_index sd={d['finlit_z_index'].std():.3f}")

Full (800): food_insecure=29.9%, very_low_security=9.9%, z_index sd=0.649
Dedup (555): food_insecure=29.2%, very_low_security=9.9%, z_index sd=0.651


## Audit Candidate Control Variables

Before selecting controls, we inspect each candidate for response coding, nonresponse
codes, and missingness — the same discipline used for the food security items. Candidates
span demographics, household composition, income/program participation, health, and housing.

In [15]:
candidate_controls = {
    "qs3":  "Age (numeric)",
    "q1":   "Education (1-5; 6=Other, 7=Refused)",
    "q3":   "Gender (1=Female, 2=Male)",
    "q4":   "Income bracket (1-14; 15=DK, 16=Refused)",
    "q5":   "Household size, others (numeric; NaN=lives alone)",
    "q23":  "Self-rated health (1-5; 6 invalid)",
    "q31":  "Housing tenure (1-4, categorical)",
    "q39":  "SNAP receipt (1=Yes,2=No,3=DK,4=Refused)",
    "q38c": "Pension income (1=Yes,2=No,3=DK,4=Refused)",
}

for var, desc in candidate_controls.items():
    print(f"--- {var}: {desc} ---")
    print(df_unique[var].value_counts(dropna=False).sort_index().to_string())
    print(f"missing: {df_unique[var].isna().sum()}\n")

--- qs3: Age (numeric) ---
qs3
60.0    10
61.0    14
62.0    14
63.0     9
64.0    11
65.0    22
66.0    19
67.0    22
68.0    26
69.0    23
70.0    27
71.0    21
72.0    35
73.0    22
74.0    21
75.0    31
76.0    23
77.0    24
78.0    20
79.0    15
80.0    24
81.0    12
82.0    19
83.0    11
84.0    10
85.0    11
86.0    14
87.0     3
88.0     4
89.0     3
90.0     7
91.0     1
92.0     1
94.0     1
95.0     1
NaN     24
missing: 24

--- q1: Education (1-5; 6=Other, 7=Refused) ---
q1
1     25
2    131
3    195
4     99
5    104
6      1
missing: 0

--- q3: Gender (1=Female, 2=Male) ---
q3
1    304
2    251
missing: 0

--- q4: Income bracket (1-14; 15=DK, 16=Refused) ---
q4
1      3
2     37
3     10
4     39
5     49
6     46
7     51
8     25
9     57
10    66
11    38
12    36
13    11
14    42
15     8
16    37
missing: 0

--- q5: Household size, others (numeric; NaN=lives alone) ---
q5
1.0     205
2.0      66
3.0      17
4.0      17
5.0       8
6.0       2
7.0       1
8.0       1

### Coding Issues Found and Recode Decisions

- **q1 (education):** codes 6 ("Other") and 7 ("Refused") are not points on the education
  ordering, so they are recoded to missing. Leaving 6 in would rank "Other" above a graduate degree.
- **q4 (income):** codes 15 ("Don't Know") and 16 ("Refused") are nonresponse → missing
  (consistent with notebook 02).
- **q23 (self-rated health):** a single value of 6 is outside the valid 1–5 range → missing.
- **q3 (gender):** recoded to a 0/1 indicator (`female` = 1).
- **q5 (household size):** missing values are NOT nonresponse — they are respondents who live
  alone. Verified against the "I live by myself" indicator (q5xR1) below, then recoded to 0.
- **q39 (SNAP):** flagged as a potential "bad control" and excluded from the main model (see below).

In [16]:
# Confirm every missing q5 is a respondent who lives alone (q5xR1 == 1)
print(pd.crosstab(df_unique["q5"].isna(), df_unique["q5xR1"].fillna(0) == 1,
                  rownames=["q5 is NaN"], colnames=["lives alone (q5xR1==1)"]))

lives alone (q5xR1==1)  False  True 
q5 is NaN                           
False                     318      0
True                        0    237


In [17]:
def clean_controls(df):
    df = df.copy()
    df["q1_clean"]      = df["q1"].replace({6: np.nan, 7: np.nan})   # drop "Other"/"Refused"
    df["q4_clean"]      = df["q4"].replace({15: np.nan, 16: np.nan}) # drop DK/Refused
    df["q23_clean"]     = df["q23"].replace({6: np.nan})            # drop invalid code
    df["female"]        = (df["q3"] == 1).astype(int)               # gender dummy
    df["hhsize_others"] = df["q5"].fillna(0)                        # lives alone -> 0 others
    return df

df_full   = clean_controls(df_full)
df_unique = clean_controls(df_unique)

chk = df_unique
print("q1_clean NaN:", chk["q1_clean"].isna().sum())
print("q4_clean NaN:", chk["q4_clean"].isna().sum())
print("q23_clean NaN:", chk["q23_clean"].isna().sum())
print("female=1:", chk["female"].sum(), "| female=0:", (chk["female"]==0).sum())
print("hhsize_others NaN:", chk["hhsize_others"].isna().sum(),
      "| lives alone (=0):", (chk["hhsize_others"]==0).sum())

q1_clean NaN: 1
q4_clean NaN: 45
q23_clean NaN: 1
female=1: 304 | female=0: 251
hhsize_others NaN: 0 | lives alone (=0): 237


### Final Control Set

Controls are chosen because they plausibly relate to BOTH financial literacy and food
insecurity (i.e., they are confounders):

- `qs3` — age
- `q1_clean` — education
- `q4_clean` — household income
- `hhsize_others` — household size
- `q23_clean` — self-rated health
- `female` — gender
- `C(q31)` — housing tenure (own-with-mortgage / own-clear / rent / no-rent)

**Excluded — SNAP (q39):** SNAP participation is a *consequence* of low income and food
hardship, not a pre-existing trait. Controlling for it risks absorbing part of the very
relationship we are studying (a "bad control"), so it is left out of the main specification
and can be revisited as a sensitivity check.

Education, income, and health are treated as continuous (ordered) for simplicity and
comparability with notebook 02; gender and housing tenure enter as categorical indicators.

## Final Adjusted Models on Both Samples

We re-estimate the adjusted models — linear probability and logit (average marginal effects) —
for both outcomes and both financial literacy measures, on both samples. Estimation
sample sizes (N) are printed because nonresponse in the controls reduces the usable sample.

In [18]:
# Diagnostic 1: multicollinearity (VIF) on the adjusted design, deduplicated sample
def vif_table(d, measure):
    cols = [measure, "qs3", "q1_clean", "q4_clean", "hhsize_others", "q23_clean", "female"]
    X = pd.get_dummies(d[cols + ["q31"]].dropna(), columns=["q31"], drop_first=True).astype(float)
    vifs = {}
    for c in X.columns:
        r2 = LinearRegression().fit(X.drop(columns=[c]), X[c]).score(X.drop(columns=[c]), X[c])
        vifs[c] = round(1 / (1 - r2), 2)
    return pd.Series(vifs, name="VIF")

print("VIF (deduplicated sample, finlit_z_index model):")
print(vif_table(df_unique, "finlit_z_index").to_string())

VIF (deduplicated sample, finlit_z_index model):
finlit_z_index    1.15
qs3               1.09
q1_clean          1.24
q4_clean          1.40
hhsize_others     1.06
q23_clean         1.16
female            1.05
q31_2             1.37
q31_3             1.55
q31_4             1.11


In [19]:
# Diagnostic 2: missing-data / selection check
# Do respondents who refused/DK income differ from those who answered?
d = df_unique
answered = d["q4"].between(1, 14)
print("Food insecurity by income-response status (deduplicated):")
print(f"  Answered income (n={answered.sum()}):    {d.loc[answered,'food_insecure'].mean():.1%}")
print(f"  DK/Refused income (n={(~answered).sum()}):  {d.loc[~answered,'food_insecure'].mean():.1%}")

Food insecurity by income-response status (deduplicated):
  Answered income (n=510):    30.6%
  DK/Refused income (n=45):  13.3%


In [20]:
samples  = {"Full (800)": df_full, "Dedup (555)": df_unique}
outcomes = ["food_insecure", "very_low_security"]
measures = ["finlit_z_index", "finlit_pca1"]
controls = "qs3 + q1_clean + q4_clean + hhsize_others + q23_clean + female + C(q31)"

print("ADJUSTED LINEAR PROBABILITY MODELS")
for sname, d in samples.items():
    print(f"\n{sname}\n" + "="*52)
    for outcome in outcomes:
        print(f"\nOutcome: {outcome}")
        for m in measures:
            model = smf.ols(f"{outcome} ~ {m} + {controls}", data=d).fit(cov_type="HC1")
            print(f"  {m:15s} coef={model.params[m]:+.3f}  p={model.pvalues[m]:.3f}  N={int(model.nobs)}")

ADJUSTED LINEAR PROBABILITY MODELS

Full (800)

Outcome: food_insecure


  finlit_z_index  coef=-0.004  p=0.886  N=712
  finlit_pca1     coef=+0.001  p=0.879  N=712

Outcome: very_low_security
  finlit_z_index  coef=-0.039  p=0.062  N=712
  finlit_pca1     coef=-0.011  p=0.101  N=712

Dedup (555)

Outcome: food_insecure
  finlit_z_index  coef=-0.052  p=0.108  N=490
  finlit_pca1     coef=-0.015  p=0.162  N=490

Outcome: very_low_security
  finlit_z_index  coef=-0.061  p=0.017  N=490
  finlit_pca1     coef=-0.019  p=0.025  N=490


In [21]:
print("ADJUSTED LOGIT — AVERAGE MARGINAL EFFECTS")
for sname, d in samples.items():
    print(f"\n{sname}\n" + "="*52)
    for outcome in outcomes:
        print(f"\nOutcome: {outcome}")
        for m in measures:
            model = smf.logit(f"{outcome} ~ {m} + {controls}", data=d).fit(disp=False)
            ame = model.get_margeff(at="overall").summary_frame()
            print(f"  {m:15s} AME={ame.loc[m,'dy/dx']:+.3f}  "
                  f"p={ame.loc[m,'Pr(>|z|)']:.3f}  N={int(model.nobs)}")

ADJUSTED LOGIT — AVERAGE MARGINAL EFFECTS

Full (800)

Outcome: food_insecure
  finlit_z_index  AME=+0.001  p=0.981  N=712
  finlit_pca1     AME=+0.003  p=0.749  N=712

Outcome: very_low_security
  finlit_z_index  AME=-0.038  p=0.017  N=712
  finlit_pca1     AME=-0.011  p=0.034  N=712

Dedup (555)

Outcome: food_insecure
  finlit_z_index  AME=-0.048  p=0.084  N=490
  finlit_pca1     AME=-0.014  p=0.132  N=490

Outcome: very_low_security
  finlit_z_index  AME=-0.055  p=0.003  N=490
  finlit_pca1     AME=-0.017  p=0.006  N=490


In [22]:
import pandas as pd

samples  = {"Full sample": df_full, "Deduplicated sample": df_unique}
outcomes = ["food_insecure", "very_low_security"]
measures = ["finlit_z_index", "finlit_pca1"]
controls = "qs3 + q1_clean + q4_clean + hhsize_others + q23_clean + female + C(q31)"

def run_one(d, outcome, rhs, m, estimator):
    if estimator == "LPM":
        res = smf.ols(f"{outcome} ~ {rhs}", data=d).fit(cov_type="HC1")
        return res.params[m], res.bse[m], res.pvalues[m], int(res.nobs)
    else:  # Logit, reported as average marginal effect
        res = smf.logit(f"{outcome} ~ {rhs}", data=d).fit(disp=False, maxiter=100)
        ame = res.get_margeff(at="overall").summary_frame()
        return ame.loc[m, "dy/dx"], ame.loc[m, "Std. Err."], ame.loc[m, "Pr(>|z|)"], int(res.nobs)

rows = []
for label, has_controls in [("Baseline", False), ("Controlled", True)]:
    for estimator in ["LPM", "Logit AME"]:
        for sname, d in samples.items():
            for outcome in outcomes:
                for m in measures:
                    rhs = f"{m} + {controls}" if has_controls else m
                    est, se, p, n = run_one(d, outcome, rhs, m, estimator)
                    rows.append({
                        "specification": f"{label} {estimator}", "sample": sname,
                        "outcome": outcome, "finlit_measure": m,
                        "estimate": est, "se": se, "p_value": p, "n_obs": n
                    })

model_summary_table = pd.DataFrame(rows).round({"estimate": 3, "se": 3, "p_value": 3})
model_summary_table

,specification,sample,outcome,finlit_measure,estimate,se,p_value,n_obs
0,Baseline LPM,Full sample,food_insecure,finlit_z_index,-0.113,0.026,0.000,800
1,Baseline LPM,Full sample,food_insecure,finlit_pca1,-0.034,0.009,0.000,800
2,Baseline LPM,Full sample,very_low_security,finlit_z_index,-0.083,0.020,0.000,800
3,Baseline LPM,Full sample,very_low_security,finlit_pca1,-0.025,0.006,0.000,800
4,Baseline LPM,Deduplicated sample,food_insecure,finlit_z_index,-0.158,0.030,0.000,555
5,Baseline LPM,Deduplicated sample,food_insecure,finlit_pca1,-0.049,0.010,0.000,555
6,Baseline LPM,Deduplicated sample,very_low_security,finlit_z_index,-0.101,0.024,0.000,555
7,Baseline LPM,Deduplicated sample,very_low_security,finlit_pca1,-0.031,0.008,0.000,555
8,Baseline Logit AME,Full sample,food_insecure,finlit_z_index,-0.106,0.023,0.000,800
9,Baseline Logit AME,Full sample,food_insecure,finlit_pca1,-0.032,0.008,0.000,800


## Person 2 Summary / Handoff (Updated — aligned with notebook 02)

**Financial literacy measures:** `finlit_z_index` (primary) and `finlit_pca1`
(alternative), matching notebook 02's locked-in names. The two correlate ~0.998,
so they are effectively one measure built two ways.

**Recoding decisions applied:**
- q1 education: "Other"/"Refused" (6, 7) → missing
- q4 income: "Don't Know"/"Refused" (15, 16) → missing
- q23 health: invalid code 6 → missing
- q3 gender → `female` dummy
- q5 household size: living-alone (verified via q5xR1) → 0 others

**Final controls:** age, education, income, household size, self-rated health,
gender, housing tenure — ordinal controls treated as continuous, which avoids the
sparse-category separation that broke the controlled logit in notebook 02.
SNAP (q39) excluded as a likely bad control.

**Diagnostics:** all VIFs well below 5 (no multicollinearity). Income non-response
is non-random — refusers are less food insecure (~13%) than answerers (~31%) — a
selection caveat for the limitations section.

**Models:** adjusted LPM (HC1 robust SEs) and logit AMEs, both outcomes, both
measures, both samples. Usable N ≈ 712/800 (full) and ≈ 490/555 (dedup).

**Open question for the team:** the control specification still differs from
notebook 02 (this notebook uses an expanded continuous set; notebook 02 uses 4
categorical controls). Align on one before freezing the final models.